In [7]:
%env AWS_PROFILE=platform-developer

env: AWS_PROFILE=platform-developer


In [ ]:
import typing
import boto3

from utils.types import FullGraphRemoverType
import config

BUCKET = config.CATALOGUE_GRAPH_S3_BUCKET
MONTHLY_TRANSFORMER_TYPES = typing.get_args(FullGraphRemoverType)

s3 = boto3.client('s3')

def copy_monthly_bulk_load_files(src_prefix: str, dst_prefix: str) -> None:
    """Copy monthly pipeline bulk load CSV files from one S3 prefix to another.

    Only files whose names start with a FullGraphRemoverType transformer type
    (i.e. LoC, MeSH, Wikidata, and WeCo) are copied.
    """
    src_prefix = src_prefix.lstrip('/')
    dst_prefix = dst_prefix.lstrip('/')

    paginator = s3.get_paginator('list_objects_v2')
    pages = paginator.paginate(Bucket=BUCKET, Prefix=src_prefix, Delimiter="/")

    copied, skipped = 0, 0
    for page in pages:
        for obj in page.get('Contents', []):
            src_key = obj['Key']
            filename = src_key[len(src_prefix):]
            if not filename.endswith('.csv'):
                skipped += 1
                continue
            if not any(filename.startswith(t) for t in MONTHLY_TRANSFORMER_TYPES):
                skipped += 1
                continue

            dst_key = dst_prefix + filename
            s3.copy_object(
                CopySource={'Bucket': BUCKET, 'Key': src_key},
                Bucket=BUCKET,
                Key=dst_key,
            )
            print(f'Copied: {filename}')
            copied += 1

    print(f'Done. {copied} file(s) copied, {skipped} skipped.')

In [ ]:
source = 'graph-prod/pipeline-2025-10-02/graph_bulk_loader/full/'
destination = 'graph-dev/pipeline-2025-10-02/graph_bulk_loader/full/'

copy_monthly_bulk_load_files(source, destination)